# 📊 Phân tích Dataset NF-UNSW-NB15-v3 (Merged)

**Mục tiêu:**
- Phân tích phân bố các loại tấn công
- Vẽ biểu đồ phân bố chi tiết
- Xác định các loại tấn công dễ nhầm lẫn
- Chuẩn chỉnh dữ liệu cho ML/DL
- Phân tích mối quan hệ giữa features

---

## ⚙️ Bước 0 — Cài đặt & Import thư viện

In [ ]:
import subprocess, sys, importlib

def ensure(pkg, import_as=None):
    name = import_as or pkg
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"✅ Cài: {pkg}")
    else:
        print(f"✅ {pkg} sẵn sàng")

for pkg in ["pandas", "numpy", "matplotlib", "seaborn", "scikit-learn", "pyarrow"]:
    ensure(pkg)

In [ ]:
import os, gc, warnings
from pathlib import Path
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns
import psutil

from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics.pairwise import euclidean_distances

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Tất cả thư viện sẵn sàng!")

## 📋 Bước 1 — Cấu hình & Load Dataset

In [ ]:
DATASET_PATH = Path("/kaggle/input/nf-uq-nids-v3-55features/NF_merged.parquet")
OUTPUT_DIR = Path("/kaggle/working/analysis_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 1_000_000
SAMPLE_SIZE = 500_000
RANDOM_STATE = 42

print(f"📂 Dataset: {DATASET_PATH}")
print(f"📁 Output: {OUTPUT_DIR}")

if DATASET_PATH.exists():
    size_gb = DATASET_PATH.stat().st_size / 1e9
    print(f"✅ Tìm thấy dataset: {size_gb:.2f} GB")
else:
    print(f"❌ Không tìm thấy: {DATASET_PATH}")

## 📥 Bước 2 — Load & Khám phá dữ liệu

In [ ]:
pf = pq.ParquetFile(DATASET_PATH)
meta = pf.metadata

print("📊 Dataset Metadata:")
print(f"  • Số dòng: {meta.num_rows:,}")
print(f"  • Số cột: {meta.num_columns}")
print(f"  • Row groups: {meta.num_row_groups}")
print(f"  • Kích thước: {DATASET_PATH.stat().st_size / 1e9:.2f} GB")

print("\n📋 Columns:")
columns = [field.name for field in pf.schema_arrow]
for i, col in enumerate(columns, 1):
    print(f"  {i:2d}. {col}")

In [ ]:
print(f"⏳ Đang load mẫu {SAMPLE_SIZE:,} dòng...")

important_cols = [
    "FLOW_START_MILLISECONDS", "FLOW_END_MILLISECONDS",
    "IN_BYTES", "OUT_BYTES", "IN_PKTS", "OUT_PKTS",
    "FLOW_DURATION_MILLISECONDS", "PROTOCOL",
    "SRC_TO_DST_SECOND_BYTES", "DST_TO_SRC_SECOND_BYTES",
    "SRC_TO_DST_AVG_THROUGHPUT", "DST_TO_SRC_AVG_THROUGHPUT",
    "TCP_FLAGS", "CLIENT_TCP_FLAGS", "SERVER_TCP_FLAGS",
    "Label", "Attack", "source_dataset"
]

df_sample = pq.read_table(
    DATASET_PATH,
    columns=important_cols,
    memory_map=True
).to_pandas()

if len(df_sample) > SAMPLE_SIZE:
    df_sample = df_sample.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)

print(f"✅ Loaded: {len(df_sample):,} rows x {len(df_sample.columns)} cols")
print(f"\n📋 Sample data:")
print(df_sample.head())

## 🎯 Bước 3 — Phân tích phân bố tấn công

In [ ]:
print("⏳ Đang đọc toàn bộ Label & Attack...")
attacks_df = pq.read_table(
    DATASET_PATH,
    columns=["Label", "Attack", "source_dataset"]
).to_pandas()

print(f"✅ Loaded: {len(attacks_df):,} rows")

print("\n" + "="*60)
print("📊 PHÂN BỐ LABEL (Benign vs Attack)")
print("="*60)

label_counts = attacks_df['Label'].value_counts().sort_index()
label_mapping = {0: 'Benign', 1: 'Attack'}

for label in sorted(label_counts.index):
    count = label_counts[label]
    pct = count / len(attacks_df) * 100
    print(f"{label_mapping[label]:10s}: {count:>12,} ({pct:6.2f}%)")

ratio = label_counts.max() / label_counts.min()
print(f"\n⚠️ Imbalance Ratio: 1:{ratio:.2f}")

In [ ]:
print("\n" + "="*60)
print("🎯 PHÂN BỐ LOẠI TẤN CÔNG (Attack Type)")
print("="*60)

attack_counts = attacks_df['Attack'].value_counts()
print(f"\n{'Attack Type':<30} {'Count':>15} {'%':>10}")
print("-"*56)

for attack, count in attack_counts.items():
    pct = count / len(attacks_df) * 100
    print(f"{attack:<30} {count:>15,} {pct:>9.2f}%")

print(f"\n📈 Số loại tấn công: {len(attack_counts)}")

## 📈 Bước 4 — Vẽ biểu đồ phân bố

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

label_names = ['Benign', 'Attack']
colors = ['#2ecc71', '#e74c3c']

axes[0].pie(label_counts.values, labels=label_names, autopct='%1.1f%%',
           colors=colors, startangle=90, textprops={'fontsize': 12, 'weight': 'bold'})
axes[0].set_title('Label Distribution', fontsize=14, weight='bold')

axes[1].bar(label_names, label_counts.values, color=colors, edgecolor='black', linewidth=1.5)
axes[1].set_ylabel('Count', fontsize=12, weight='bold')
axes[1].set_title('Label Count', fontsize=14, weight='bold')
axes[1].grid(axis='y', alpha=0.3)

for i, v in enumerate(label_counts.values):
    axes[1].text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=11, weight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_label_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: 01_label_distribution.png")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

y_pos = np.arange(len(attack_counts))
colors_attack = plt.cm.Set3(np.linspace(0, 1, len(attack_counts)))

ax.barh(y_pos, attack_counts.values, color=colors_attack, edgecolor='black', linewidth=1.2)
ax.set_yticks(y_pos)
ax.set_yticklabels(attack_counts.index, fontsize=11)
ax.set_xlabel('Count', fontsize=12, weight='bold')
ax.set_title('Attack Type Distribution', fontsize=14, weight='bold')
ax.grid(axis='x', alpha=0.3)

for i, v in enumerate(attack_counts.values):
    ax.text(v, i, f' {v:,}', va='center', fontsize=10, weight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '02_attack_type_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: 02_attack_type_distribution.png")

## 🔍 Bước 5 — Phân tích những loại tấn công dễ nhầm lẫn

In [ ]:
print("⏳ Đang load dữ liệu cho phân tích confusion...")

numeric_cols = [
    'IN_BYTES', 'OUT_BYTES', 'IN_PKTS', 'OUT_PKTS',
    'FLOW_DURATION_MILLISECONDS', 'PROTOCOL',
    'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES',
    'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT',
    'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS'
]

df_features = pq.read_table(
    DATASET_PATH,
    columns=numeric_cols + ['Attack']
).to_pandas()

df_features_sample = df_features.sample(n=min(SAMPLE_SIZE, len(df_features)), 
                                        random_state=RANDOM_STATE).reset_index(drop=True)

print(f"✅ Loaded: {len(df_features_sample):,} rows")

In [ ]:
print("\n" + "="*70)
print("⚡ PHÂN TÍCH NHỮNG LOẠI TẤN CÔNG DỄ NHẦM LẪN")
print("="*70)

scaler = RobustScaler()
X_normalized = scaler.fit_transform(df_features_sample[numeric_cols].fillna(0))

attack_types = df_features_sample['Attack'].unique()
attack_centroids = {}

for attack in attack_types:
    mask = df_features_sample['Attack'] == attack
    if mask.sum() > 0:
        attack_centroids[attack] = X_normalized[mask].mean(axis=0)

distances = {}
for i, attack1 in enumerate(attack_types):
    for attack2 in attack_types[i+1:]:
        if attack1 in attack_centroids and attack2 in attack_centroids:
            dist = euclidean_distances(
                [attack_centroids[attack1]], 
                [attack_centroids[attack2]]
            )[0][0]
            distances[(attack1, attack2)] = dist

sorted_distances = sorted(distances.items(), key=lambda x: x[1])

print("\n🔴 Top 10 cặp tấn công CÓ KHUYNH HƯỚNG NHẦM LẪN:")
for i, ((attack1, attack2), dist) in enumerate(sorted_distances[:10], 1):
    count1 = (df_features_sample['Attack'] == attack1).sum()
    count2 = (df_features_sample['Attack'] == attack2).sum()
    print(f"\n  {i}. {attack1} <-> {attack2}")
    print(f"     Khoảng cách: {dist:.4f}")
    print(f"     Mẫu: {attack1}={count1:,}, {attack2}={count2:,}")

## 🔧 Bước 6 — Chuẩn chỉnh dữ liệu cho ML/DL

In [ ]:
print("="*70)
print("📋 PHƯƠNG PHÁP CHUẨN CHỈNH DỮ LIỆU")
print("="*70)

print("\n1️⃣ STANDARDIZATION (Z-score):")
print("   • Công thức: (x - mean) / std")
print("   • Khi dùng: Linear models, SVM, KNN")

scaler_standard = StandardScaler()
X_standard = scaler_standard.fit_transform(df_features_sample[numeric_cols].fillna(0))

print("\n2️⃣ MIN-MAX SCALING:")
print("   • Công thức: (x - min) / (max - min)")
print("   • Khi dùng: Neural networks")

scaler_minmax = MinMaxScaler()
X_minmax = scaler_minmax.fit_transform(df_features_sample[numeric_cols].fillna(0))

print("\n3️⃣ ROBUST SCALING:")
print("   • Công thức: (x - median) / IQR")
print("   • Khi dùng: Dữ liệu có outliers")

X_robust = scaler.fit_transform(df_features_sample[numeric_cols].fillna(0))

print("\n✅ Tất cả phương pháp đã chuẩn bị")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

feature_to_plot = 'IN_BYTES'
idx = numeric_cols.index(feature_to_plot)

axes[0, 0].hist(df_features_sample[feature_to_plot].fillna(0), bins=50, color='blue', alpha=0.7)
axes[0, 0].set_title('Original Data', fontsize=12, weight='bold')

axes[0, 1].hist(X_standard[:, idx], bins=50, color='green', alpha=0.7)
axes[0, 1].set_title('StandardScaler', fontsize=12, weight='bold')

axes[1, 0].hist(X_minmax[:, idx], bins=50, color='orange', alpha=0.7)
axes[1, 0].set_title('MinMaxScaler', fontsize=12, weight='bold')

axes[1, 1].hist(X_robust[:, idx], bins=50, color='red', alpha=0.7)
axes[1, 1].set_title('RobustScaler', fontsize=12, weight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_scaling_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: 03_scaling_comparison.png")

## 🎨 Bước 7 — Phân tích PCA

In [ ]:
print("\n" + "="*70)
print("📊 PRINCIPAL COMPONENT ANALYSIS (PCA)")
print("="*70)
print("\n⏳ Chạy PCA...")

pca = PCA()
pca_result = pca.fit_transform(X_standard)
cumsum_var = np.cumsum(pca.explained_variance_ratio_)

print(f"\nVariance explained (top 10):")
for i in range(min(10, len(pca.explained_variance_ratio_))):
    print(f"  PC{i+1}: {pca.explained_variance_ratio_[i]:.4f} (cumsum: {cumsum_var[i]:.4f})")

n_components_95 = np.argmax(cumsum_var >= 0.95) + 1
print(f"\n💡 Components cho 95% variance: {n_components_95} (từ {len(numeric_cols)} features)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, len(pca.explained_variance_ratio_) + 1), 
           pca.explained_variance_ratio_, alpha=0.7, color='steelblue')
axes[0].set_xlabel('Principal Component', fontsize=12, weight='bold')
axes[0].set_ylabel('Variance Explained', fontsize=12, weight='bold')
axes[0].set_title('PCA Scree Plot', fontsize=14, weight='bold')
axes[0].grid(axis='y', alpha=0.3)

axes[1].plot(range(1, len(cumsum_var) + 1), cumsum_var, 'bo-', linewidth=2)
axes[1].axhline(y=0.95, color='r', linestyle='--', linewidth=2, label='95% threshold')
axes[1].axvline(x=n_components_95, color='g', linestyle='--', linewidth=2)
axes[1].set_xlabel('Components', fontsize=12, weight='bold')
axes[1].set_ylabel('Cumulative Variance', fontsize=12, weight='bold')
axes[1].set_title('Cumulative Variance Explained', fontsize=14, weight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_pca_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: 04_pca_analysis.png")

## 📄 Bước 8 — Tạo báo cáo tổng hợp

In [ ]:
report = f"""
╔════════════════════════════════════════════════════════════════════════════╗
║           NF-UNSW-NB15-v3 DATASET ANALYSIS REPORT                         ║
╚════════════════════════════════════════════════════════════════════════════╝

📊 DATASET OVERVIEW
{'='*80}
• Total Rows: {len(attacks_df):,}
• Total Columns: {meta.num_columns}
• Size: {DATASET_PATH.stat().st_size / 1e9:.2f} GB
• Format: Apache Parquet (Snappy compression)

🏷️ LABEL DISTRIBUTION (Binary Classification)
{'='*80}
• Benign: {label_counts[0]:>15,} ({label_counts[0]/len(attacks_df)*100:6.2f}%)
• Attack: {label_counts[1]:>15,} ({label_counts[1]/len(attacks_df)*100:6.2f}%)
• Imbalance Ratio: 1:{ratio:.2f}

🎯 TOP 10 ATTACK TYPES
{'='*80}
"""

for idx, (attack, count) in enumerate(attack_counts.head(10).items(), 1):
    report += f"  {idx:2d}. {attack:<30} {count:>12,}\n"

report += f"""

⚡ CONFUSION ANALYSIS (Top 5 Similar Attack Pairs)
{'='*80}
"""

for i, ((attack1, attack2), dist) in enumerate(sorted_distances[:5], 1):
    report += f"""
  {i}. {attack1} <-> {attack2}
     Distance: {dist:.4f}
     Risk: Classifier may confuse these attack types
"""

report += f"""

🔧 DATA PREPROCESSING RECOMMENDATIONS
{'='*80}

1. SCALING METHODS:
   ✓ StandardScaler: For Linear models, SVM, KNN
   ✓ MinMaxScaler: For Neural Networks
   ✓ RobustScaler: For outlier-prone data

2. CLASS IMBALANCE:
   • Use SMOTE (Synthetic Minority Oversampling)
   • Apply class weights in loss function
   • Use F1-Score, AUC-ROC for evaluation

3. DIMENSIONALITY REDUCTION:
   • PCA: {n_components_95} components for 95% variance
   • Reduction: {(1 - n_components_95/len(numeric_cols))*100:.1f}%

📅 Report Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
╚════════════════════════════════════════════════════════════════════════════╝
"""

with open(OUTPUT_DIR / 'analysis_report.txt', 'w', encoding='utf-8') as f:
    f.write(report)

print(report)
print("\n✅ Saved: analysis_report.txt")

## ✅ Hoàn tất!

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════════════════╗
║                    ANALYSIS COMPLETE! ✅                                    ║
╚════════════════════════════════════════════════════════════════════════════╝

📊 Generated Files:
   • 01_label_distribution.png
   • 02_attack_type_distribution.png
   • 03_scaling_comparison.png
   • 04_pca_analysis.png
   • analysis_report.txt

📁 Output Directory: /kaggle/working/analysis_output/

🎯 Key Insights:
   1. Analyzed label distribution and class imbalance
   2. Identified similar attack types using Euclidean distance
   3. Compared 3 scaling methods for preprocessing
   4. Performed PCA for dimensionality reduction
   5. Generated comprehensive analysis report

""")

print(f"RAM Usage: {psutil.virtual_memory().percent}%")